### Research Question

This paper investigates whether higher labour market coordination is associated with greater labour market resilience in advanced economies from 2000–2023 where labour market resilience is measured using changes in unemployment rates across countries and over time.

This Question seeks to uncover the **Descriptive** relationship between the variables COORD and Unemployment, controlling for other variables. This is because COORD is an institutional trait, rather than a choice or treatment; many macroeconomic variables exist and are ommited for simplicity sake, and there exists no clean identification for isolating the effects of COORD to Unemployment, only its pure association. A causal interpretation would require stronger identification assumptions regarding the exogeneity of labour market coordination institutions, which are unlikely to hold in observational macroeconomic panel data.

In [11]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

panel = pd.read_csv("data/clean/final_panel.csv")

display(panel.head())

,Country,Year,Unemployment,Delta_Unemployment,COORD,GDPGrowth,Inflation,ExportDependence,GovSpend
0,Australia,2000,6.288,NaN,2.0,3.916057,4.457437,19.356083,18.505430
1,Australia,2001,6.747,0.459,2.0,2.031459,4.407130,22.109757,18.507007
2,Australia,2002,6.375,-0.372,2.0,3.949805,2.981571,20.692768,18.318430
3,Australia,2003,5.933,-0.442,2.0,3.091280,2.732598,19.025563,18.380148
4,Australia,2004,5.399,-0.534,2.0,4.218941,2.343257,17.135063,18.273056


### Regression Equation

The baseline specification is estimated using panel-data ordinary least squares with country and year fixed effects.

The estimated model is:

ΔUnemployment_it =
β0 + β1COORD_it + β2GDPGrowth_it + β3Inflation_it + β4ExportDependence_it + β5GovSpend_it + α_i + γ_t + ε_it

Where:

- ΔUnemployment_it is the annual change in unemployment rate
- COORD_it is the labour market coordination index
- α_i represents country fixed effects
- γ_t represents year fixed effects
- ε_it is the error term

Country fixed effects control for time-invariant country characteristics, while year fixed effects control for common macroeconomic shocks affecting all countries. FE's are included because advanced economies differ in persistent institutional and structural characteristics such as welfare systems, labour regulations, demographics, and industrial composition that may influence unemployment outcomes.

Year fixed effects are included to control for global shocks affecting all countries simultaneously, including the Global Financial Crisis and COVID-19.

[Note***: The dependent variable is specified as the annual change in unemployment (Delta_unemployment) rather than the unemployment level itself. This focuses the analysis on labour market resilience and short-run labour market adjustments during economic shocks rather than persistent long-run differences in unemployment levels across countries.]

### Regression Code

In [21]:
import pandas as pd
import statsmodels.formula.api as smf

reg_data = pd.read_csv("data/clean/final_panel.csv")

model = smf.ols(
    formula="""
    Delta_Unemployment ~
    COORD +
    GDPGrowth +
    Inflation +
    ExportDependence +
    GovSpend +
    C(Country) +
    C(Year)
    """,
    data=reg_data
).fit(cov_type="HC3")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:     Delta_Unemployment   R-squared:                       0.522
Model:                            OLS   Adj. R-squared:                  0.460
Method:                 Least Squares   F-statistic:                     7.866
Date:                Thu, 14 May 2026   Prob (F-statistic):           4.57e-30
Time:                        11:04:33   Log-Likelihood:                -289.96
No. Observations:                 365   AIC:                             665.9
Df Residuals:                     322   BIC:                             833.6
Df Model:                          42                                         
Covariance Type:                  HC3                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [24]:
from statsmodels.iolib.summary2 import summary_col

table = summary_col(
    [model],
    stars=True,
    model_names=["Main Specification"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R²": lambda x: f"{x.rsquared:.4f}"
    }
)

print(table)


                             Main Specification
-----------------------------------------------
Intercept                    -1.9384*          
                             (1.0187)          
C(Country)[T.Austria]        -0.1876           
                             (0.5233)          
C(Country)[T.Belgium]        -0.7561           
                             (0.8533)          
C(Country)[T.Canada]         -0.3294           
                             (0.3005)          
C(Country)[T.Denmark]        -0.8398           
                             (0.6584)          
C(Country)[T.Finland]        -0.7564           
                             (0.4704)          
C(Country)[T.Germany]        -0.3430           
                             (0.4680)          
C(Country)[T.Japan]          -0.1776           
                             (0.4384)          
C(Country)[T.Netherlands]    -0.8274           
                             (0.7921)          
C(Country)[T.New Zealand]    -0.0444   

In [28]:
import os

os.makedirs("output", exist_ok=True)

with open("output/main_regression_table.txt", "w") as f:
    f.write(table.as_text())

## Interpretation of Regression Results

- COORD

The coefficient on COORD is -0.0501. This suggests that a one-unit increase in labour market coordination is associated with an approximately 0.05 percentage-point decrease in annual unemployment growth, holding GDP growth, inflation, export dependence, government spending, country fixed effects, and year fixed effects constant. However, the standard error(SE) for COORD is relatively large (0.1601), indicating that the estimated relationship is statistically weak and imprecise. This suggests that labour market coordination alone does not appear to strongly explain unemployment resilience once broader macroeconomic conditions and fixed effects are controlled for.

The coefficient, in itself, is not statistically significant at the 5% level (p > 0.05). Therefore, H0, that the COORD coefficient equals zero cannot be rejected. This suggests that the analysis does not provide strong statistical evidence that labour market coordination is systematically associated with unemployment resilience after controlling for macroeconomic variables and fixed effects. 

One possible explanation for the weak statistical relationship between COORD and unemployment resilience is that labour market coordination is relatively time-invariant within countries. Because the model includes country fixed effects, much of the long-run institutional variation across countries may already be absorbed by the country-specific intercepts. As a result, the fixed-effects specification identifies the COORD relationship primarily through within-country changes over time, which are relatively limited in the sample. This may reduce the precision and statistical significance of the estimated COORD coefficient. This will be further tested upon in the Robustness Checks.

Otherwise, the direction of the relationship still remains broadly consistent with the theoretical expectation that more coordinated labour market systems may smooth unemployment adjustments during economic shocks.

- GDP Growth

GDP growth has a negative coefficient (-0.1135), suggesting that stronger economic growth is associated with lower unemployment growth as expected to the theorethical economic result. Specifically, a one percentage-point increase in GDP growth is associated with approximately a 0.11 percentage-point decrease in annual unemployment growth, holding other variables constant. 

- Inflation

Inflation also has a negative estimated relationship with unemployment growth (-0.0446), although the effect appears relatively modest, reflecting short-run macroeconomic trade-offs between inflation and unemployment.

- Export Dependence

Export dependence has a very small positive coefficient (0.0065), suggesting little meaningful association between export exposure and annual unemployment changes within this specification.

- Government Spending

Government spending has a positive coefficient (0.1191), indicating that higher government expenditure is associated with higher unemployment growth in the sample. However, this relationship likely reflects governments increasing expenditure during periods of economic stress and rising unemployment rather than government spending directly causing unemployment increases.

The model includes both country fixed effects and year fixed effects. Country fixed effects control for time-invariant differences between countries such as institutional structures, geography, and long-run economic systems. Year fixed effects control for global shocks affecting all countries simultaneously, such as the Global Financial Crisis and COVID-19.

The model explains approximately 52.2% of the variation in annual unemployment changes (R² = 0.522), indicating moderate explanatory power for a macroeconomic panel-data specification.

Overall, the results display limited evidence that higher labour market coordination is strongly associated with improved labour market resilience once broader macroeconomic controls and fixed effects are included.

### Threats and Limitations

The main limitation is omitted confounding. Countries with higher labour market coordination may also differ in welfare systems, union structures, employment protection laws, fiscal capacity, industrial composition, and political institutions. These factors may also affect unemployment resilience. Therefore, country fixed effects were established to reduce bias from time-invariant country characteristics, and year fixed effects control for global shocks affecting all countries. However, time-varying omitted variables may still remain --> Leading to Omitted Variable Bias (OVB)

Secondly, reverse causality may also be relevant if persistent labour market conditions influence institutional reforms and labour market coordination structures over time. This analysis is **Descriptive**, however, this is a possibility in any words.

Additionally, as the sample is limited to advanced economies with relatively stable institutions and higher data availability. Hence, the results may not generalize to developing economies or countries with substantially different labour market structures.

### Conclusion

To conclude, the model produced from the datasets indicate this model has moderate explanatory power, however, results are limited as to whether COORD is highly associated with better labour market resilience. COORD has an overall positive effect as the model had expected, but also remains mostly statistcally insignificant under the 5% significance level. This is potentially due to country FE's absorbing variation within the variable, which will be tested further in the Robustness Checks.

As Institutional variables are often difficult to estimate precisely in fixed-effects panel models because institutional structures evolve slowly relative to macroeconomic fluctuations.